# DARWIN-1 environment-dependent UV luminosity function

Example notebook for `darwin1_uvlf.py` (Kang et al., MNRAS, in revision).

`uvlf(M_UV, z, delta, form)` returns Φ(M_UV) in cMpc⁻³ mag⁻¹ for a region of matter overdensity δ (defined on a cube of side ≈ 8–9 cMpc, i.e. one NIRCam pointing) at redshift z. Calibrated range: 6 ≲ z ≲ 15, −0.35 ≲ δ ≲ 0.35; outside it the function is an extrapolation.

Requirements: `numpy`, `matplotlib` (plots only).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from darwin1_uvlf import uvlf, params, COEF_SCH, COEF_SCH_ERR

M = np.arange(-23.0, -15.9, 0.1)

## 1. A single luminosity function

In [ ]:
z, delta = 10.0, 0.0
phi_s = uvlf(M, z, delta, form='schechter')
phi_d = uvlf(M, z, delta, form='dpl')

plt.figure(figsize=(5.5, 4))
plt.semilogy(M, phi_s, label='Schechter')
plt.semilogy(M, phi_d, '--', label='double power law')
plt.gca().invert_xaxis(); plt.xlabel('$M_{UV}$'); plt.ylabel(r'$\Phi$ [cMpc$^{-3}$ mag$^{-1}$]')
plt.title(rf'z = {z}, $\delta$ = {delta}'); plt.legend(); plt.show()

## 2. Dependence on the local overdensity

The bright end responds much more strongly to δ than the faint end: at z = 10 a region with δ = +0.2 has ≈ 2.5× more M_UV ≈ −18.5 galaxies and ≈ 10× more M_UV ≈ −21 galaxies than the cosmic mean.

In [ ]:
z = 10.0
plt.figure(figsize=(5.5, 4))
for d in (-0.3, -0.15, 0.0, 0.15, 0.3):
    plt.semilogy(M, uvlf(M, z, d), label=rf'$\delta$ = {d:+.2f}')
plt.gca().invert_xaxis(); plt.xlabel('$M_{UV}$'); plt.ylabel(r'$\Phi$ [cMpc$^{-3}$ mag$^{-1}$]'); plt.title(f'z = {z}'); plt.legend(); plt.show()

for Mq in (-18.5, -20.0, -21.0):
    print(f'M_UV = {Mq:5.1f}: Phi(delta=+0.2)/Phi(0) = {uvlf(Mq, z, 0.2)/uvlf(Mq, z, 0.0):.2f}')

## 3. Redshift evolution at fixed overdensity

In [ ]:
plt.figure(figsize=(5.5, 4))
for z in (6, 8, 10, 12, 14):
    plt.semilogy(M, uvlf(M, z, 0.0), label=f'z = {z}')
plt.gca().invert_xaxis(); plt.xlabel('$M_{UV}$'); plt.ylabel(r'$\Phi$ [cMpc$^{-3}$ mag$^{-1}$]'); plt.title(r'$\delta$ = 0'); plt.legend(); plt.show()

## 4. Schechter parameters as a function of z and δ

In [ ]:
zz = np.linspace(6, 15, 50)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for d in (-0.3, 0.0, 0.3):
    a, ms, lp = params(zz, d, 'schechter')
    axes[0].plot(zz, a, label=rf'$\delta$ = {d:+.1f}'); axes[1].plot(zz, ms); axes[2].plot(zz, lp)
for ax, lab in zip(axes, [r'$\alpha$', '$M^*$', r'$\log_{10}\phi^*$']):
    ax.set_xlabel('z'); ax.set_ylabel(lab)
axes[0].legend(); plt.tight_layout(); plt.show()

## 5. Propagating the coefficient uncertainties

`COEF_SCH_ERR` holds the 1σ uncertainties of the coefficients (uncorrelated here; the full chain is available on request). A quick Monte Carlo gives the spread of Φ at a given magnitude. Note that the intrinsic scatter of the parameterisation about the simulation (0.10 in α, 0.47 mag in M*, 0.14 dex in log φ*) is larger than the coefficient uncertainties and dominates the bright-end extrapolation.

In [ ]:
rng = np.random.default_rng(0)
z, d, Mq = 14.0, 0.0, -20.5
samples = [uvlf(Mq, z, d, coef=COEF_SCH + COEF_SCH_ERR*rng.standard_normal(COEF_SCH.shape)) for _ in range(2000)]
lo, med, hi = np.percentile(samples, [16, 50, 84])
print(f'Phi(M_UV={Mq}, z={z}, delta={d}) = {med:.2e} (+{hi-med:.1e} / -{med-lo:.1e}) cMpc^-3 mag^-1 (coefficient uncertainties only)')

## 6. Reading the coefficient table

In [ ]:
import csv
with open('coefficients.csv') as f:
    for row in csv.reader(f):
        print(', '.join(row))